# 22 Where voids meet: boundaries through open void

The distance basins (notebooks 17 and 18) split the void voxels into voids by climbing the distance to structure. Where two basins meet in open void, the boundary is a geometric cut, roughly equidistant from two stretches of wall, not a wall itself; at Δ = 0.5 h⁻¹ Mpc such cuts show up as straight lines in slices. This notebook measures how much of that there is at each Δ, for the truth and for DeepVoid's predictions, before trying a better way to split.

**What is measured.** For the voids at a given Δ (minimum radius as notebook 18):

- **Internal boundary**: void voxels with a neighbor in a different void, as opposed to the **structure boundary** (a neighbor that is wall, filament or halo). Its share of all boundary faces says how much of the void surfaces are cuts between voids at all.
- **How far internal-boundary voxels sit from structure** (the distance to structure on them), in cells: ≤ 1 cell means the cut touches structure (a real wall or a thin neck); beyond 2 cells it runs through open void.
- **Per void**: the share of each void's surface that is an open-void cut (internal boundary more than 2 cells from structure), and how many voids above R_eff = 3 h⁻¹ Mpc have more than 10% and 25% of their surface that way.

A Δ floor (merging every cut shallower than a cell or two) is the same as a larger Δ, so the rows for Δ = 1, 2, 3 show what a floor would do.

**Output.** `void_definitions/void_boundaries.json` (the tables, reused on reruns) and the figures `void_boundaries.png`, `void_boundaries_slice.png`.

**Runtime.** CPU, High-RAM. For each source (truth, DeepVoid at λ = 3 and 10): the distance to structure, basins and merge tree at 512³ (a few minutes), then a quick cut per Δ.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/deepvoid
!git clone -q "https://github.com/skumagai3/deepvoid.git" /content/deepvoid
!pip -q install --force-reinstall --no-deps /content/deepvoid
!git -C /content/deepvoid log --oneline -1

## 1 Settings

In [ ]:
import gc
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.ticker import MaxNLocator

from deepvoid import Catalog
from deepvoid import voids as V
from deepvoid.labels import VOID

RUNS = Path('/content/drive/MyDrive/deepvoid_runs')
OUT = RUNS / 'void_definitions'
OUT.mkdir(parents=True, exist_ok=True)
PRED = RUNS / 'predictions'                               # whole-box scores (notebook 17)
RESULT = OUT / 'void_boundaries.json'
TIMINGS = RUNS / 'voids' / 'timings.json'                 # shared with notebooks 17, 18, 21
catalog = Catalog()

LABELS = 'tng300-3/mask'                                  # TNG300-3, 512^3, 0.40 h^-1 Mpc
MODEL = 'subhalo_C_mixed_smooth2ch_depth5_distill'        # as notebook 18
SPACINGS = [3, 10]                                        # DeepVoid predictions to include (if saved)
DELTAS = [0.5, 1.0, 2.0, 3.0]                             # h^-1 Mpc
SIGMA_EFF = 1.0                                           # h^-1 Mpc: T-web smoothing of the labels
MIN_RADIUS = SIGMA_EFF                                    # as notebook 18: the label smoothing
MARGIN = 96
R_CUT = 3.0                                               # voids counted per void: R_eff above this
OPEN_CELLS = 2.0                                          # an internal boundary farther than this from structure is an open-void cut
DIST_BINS = [0.0, 1.01, 2.01, 4.01, np.inf]               # cells (edges just above 1, 2, 4: a voxel next to structure is at exactly 1)
DIST_LABELS = ['1 (touches structure)', '1–2', '2–4', '> 4']
SLICE = 256                                               # x index of the figure slice
SLICE_DELTA = 1.0
REFRESH = False


def timed(name, fn):
    # fn(), with how long the same step took last time (if known); records this run's time
    t = json.loads(TIMINGS.read_text()) if TIMINGS.exists() else {}
    print(f'{name}: ' + (f'last time {t[name] / 60:.1f} min' if name in t else 'no earlier timing'), flush=True)
    t0 = time.time()
    res = fn()
    dt = time.time() - t0
    t = json.loads(TIMINGS.read_text()) if TIMINGS.exists() else {}
    t[name] = dt
    TIMINGS.write_text(json.dumps(t, indent=1))
    print(f'{name}: {dt / 60:.1f} min', flush=True)
    return res

## 2 Measure

For each source and Δ: the cut, the two kinds of boundary, the distances to structure on the internal boundary, and per-void shares. Results go to `void_boundaries.json` after each source, so a rerun skips finished sources. The slice for the figure (truth and the first prediction, at `SLICE_DELTA`) is kept in memory only, so it is drawn only for sources computed in this run (`REFRESH = True` redraws it).

In [ ]:
labels = catalog.load(LABELS)
n, cell = labels.n, labels.box_size / labels.n
sources = {'T-web truth': lambda: labels.data == VOID}
for s in SPACINGS:
    f = PRED / f'tng_{MODEL}_L{s}.npz'
    if f.exists():
        sources[f'DeepVoid λ = {s}'] = (lambda f=f: np.load(f)['scores'].argmax(-1) == VOID)
    else:
        print(f'{f.name} not found: run notebook 17 with TRUTH_ONLY = False; λ = {s} left out')

results = json.loads(RESULT.read_text()) if RESULT.exists() and not REFRESH else {}
slices = {}


def measure(name):
    void = sources[name]()
    d = V.distance_to_structure(void, cell=cell, margin=MARGIN)
    seeds = V.ascent_basins(d, void)[0]
    tree = V.merge_tree(seeds, d)
    rows = {}
    for delta in DELTAS:
        lab = V.cut_tree(seeds, tree, delta, distance=d, min_radius=MIN_RADIUS)
        b = V.internal_boundaries(lab, void)
        dist_cells = d[b['internal']] / cell
        hist = np.histogram(dist_cells, DIST_BINS)[0]
        open_cut = b['internal'] & (d > (OPEN_CELLS + 0.01) * cell)
        n_lab = int(lab.max())
        surface = np.bincount(lab[b['internal'] | b['structure']], minlength=n_lab + 1)
        cut = np.bincount(lab[open_cut], minlength=n_lab + 1)
        r_eff = V.void_catalog(lab, cell)['r_eff']
        big = np.flatnonzero(r_eff > R_CUT) + 1          # void labels are 1..n
        share = cut[big] / np.maximum(surface[big], 1)
        rows[str(delta)] = {
            'voids above R_CUT': int(len(big)),
            'internal share of boundary faces': b['internal_faces'] / max(b['internal_faces'] + b['structure_faces'], 1),
            **{f'internal boundary at {DIST_LABELS[i]} cells': (float(h / hist.sum()) if hist.sum() else float('nan'))
               for i, h in enumerate(hist)},
            'median open-cut share of a void surface': float(np.median(share)) if len(share) else float('nan'),
            'voids with > 10% open-cut surface': float((share > 0.10).mean()) if len(share) else float('nan'),
            'voids with > 25% open-cut surface': float((share > 0.25).mean()) if len(share) else float('nan'),
        }
        if delta == SLICE_DELTA and len(slices) < 2:
            slices[name] = {'lab': lab[SLICE].copy(), 'void': void[SLICE].copy(),
                            'internal': b['internal'][SLICE].copy(), 'd': d[SLICE].copy()}
        del lab, b, open_cut
    del void, d, seeds, tree
    gc.collect()
    return rows


for name in sources:
    if name in results:
        print(f'{name}: from {RESULT.name}')
        continue
    rows = timed(f'void_boundaries_{name}', lambda: measure(name))
    results[name] = rows
    RESULT.write_text(json.dumps(results, indent=1, ensure_ascii=False))

table = pd.DataFrame({(src, f'Δ = {k}'): v for src, rows in results.items() for k, v in rows.items()}).T
print(table.round(3).to_string())

## 3 Figures

Left: share of the internal boundary at each distance from structure, per Δ (truth solid, predictions dashed). Right: share of voids (R_eff > 3) whose surface is more than 10% open-void cut. Below: a slice at Δ = `SLICE_DELTA`, each void in its own color, structure in white, internal boundaries in black where they touch structure (≤ 1 cell) and red where they run through open void (> 2 cells). Boundaries are 3D, so a void whose neighbor lies just above or below the slice shows red inside it, not only along the edges seen in the slice. With no boundaries between voids at some Δ (one void per region), its line on the left is left out.

In [ ]:
srcs = list(results)
styles = {s: ('-' if s == 'T-web truth' else '--' if s.endswith('= 3') else ':') for s in srcs}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
bins = DIST_LABELS
colors = dict(zip(DELTAS, ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']))
for s in srcs:
    for delta in DELTAS:
        r = results[s][str(delta)]
        axes[0].plot(range(len(bins)), [r[f'internal boundary at {b} cells'] for b in bins], styles[s],
                     color=colors[delta], marker='o', ms=3)
    axes[1].plot(DELTAS, [results[s][str(dl)]['voids with > 10% open-cut surface'] for dl in DELTAS],
                 styles[s], color='k', marker='o', ms=4)
axes[0].set_xticks(range(len(bins)), bins, rotation=15)
axes[0].set_xlabel('distance from the internal-boundary voxel to structure (cells of 0.40 h$^{-1}$ Mpc)')
axes[0].set_ylabel('share of internal-boundary voxels')
axes[0].set_title('Where voids meet: distance from structure')
axes[1].set_xlabel('merge threshold Δ (h$^{-1}$ Mpc)')
axes[1].set_ylabel('share of voids (R$_{eff}$ > 3 h$^{-1}$ Mpc)')
axes[1].set_title('Voids with > 10% of their surface cut through open void')
for ax in axes:
    ax.set_ylim(bottom=0)
    ax.yaxis.set_major_locator(MaxNLocator(steps=[1, 2, 5, 10]))
handles = [plt.Line2D([], [], color=c, label=f'Δ = {dl} h$^{{-1}}$ Mpc (left panel)') for dl, c in colors.items()]
handles += [plt.Line2D([], [], color='gray', ls=styles[s], label=s) for s in srcs]
fig.legend(handles=handles, loc='center left', bbox_to_anchor=(1.0, 0.5), frameon=False)
fig.suptitle('Void boundaries between voids, TNG300-3 (512$^3$): how many run through open void')
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.savefig(OUT / 'void_boundaries.png', dpi=150, bbox_inches='tight')
plt.show()

if slices:
    rng = np.random.default_rng(0)
    fig, axes = plt.subplots(1, len(slices), figsize=(7 * len(slices), 7), squeeze=False)
    for ax, (s, sl) in zip(axes[0], slices.items()):
        lab = sl['lab']
        cmap = ListedColormap(np.vstack([[1, 1, 1, 1], np.column_stack([rng.uniform(0.35, 0.9, (lab.max(), 3)), np.ones(lab.max())])]))
        ext = [0, n * cell, 0, n * cell]
        ax.imshow(lab.T, origin='lower', cmap=cmap, interpolation='nearest', extent=ext)
        touch = sl['internal'] & (sl['d'] <= 1.01 * cell)
        open_ = sl['internal'] & (sl['d'] > (OPEN_CELLS + 0.01) * cell)
        for m, c in ((touch, 'k'), (open_, 'red')):
            ov = np.zeros(m.shape + (4,))
            ov[m] = plt.matplotlib.colors.to_rgba(c)
            ax.imshow(ov.transpose(1, 0, 2), origin='lower', interpolation='nearest', extent=ext)
        ax.set_title(f'{s}, Δ = {SLICE_DELTA} h$^{{-1}}$ Mpc, x = {SLICE * cell:.1f} h$^{{-1}}$ Mpc')
        ax.set_xlabel('y (h$^{-1}$ Mpc)')
        ax.set_ylabel('z (h$^{-1}$ Mpc)')
    handles = [plt.Line2D([], [], color='k', lw=3, label='boundary between voids, touching structure (≤ 1 cell)'),
               plt.Line2D([], [], color='red', lw=3, label=f'boundary between voids through open void (> {OPEN_CELLS:g} cells)'),
               plt.Line2D([], [], color='lightgray', lw=6, label='one color per void; white = wall, filament, halo')]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.02), frameon=False, ncol=1)
    fig.suptitle('Voids in a slice and where they meet each other')
    fig.tight_layout(rect=(0, 0.06, 1, 0.95))
    fig.savefig(OUT / 'void_boundaries_slice.png', dpi=150, bbox_inches='tight')
    plt.show()

## Shut down

Deletes this runtime once every cell above has finished, so it stops using compute units. The results are already on Drive. With **Run all**, Colab stops at the first cell that fails, so after an error the runtime stays up and you can look at it. Set `SHUT_DOWN = False` to keep the runtime.

In [ ]:
SHUT_DOWN = True   # False keeps the runtime after the run
if SHUT_DOWN:
    from google.colab import drive, runtime
    drive.flush_and_unmount()   # finish writing to Drive first
    time.sleep(5)
    runtime.unassign()          # disconnects and deletes this runtime